In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from plotnine import *

In [ ]:
anno_dir = 'PATH_TO_FILE'

# Add UKBBGym annotations

1. VEP coding / non-coding
2. Amino acid position
3. Protein domains (MOBI db)

In [ ]:
# Download variant metadata file
!dx download project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated_variant_metadata.parquet -o {anno_dir}

varmeta_df = pl.read_parquet(f'{anno_dir}/qced_maf1e-3_loftee_olink_genes_EURunrelated_variant_metadata.parquet')
varmeta_df.head()

In [ ]:
# Download annotations file
!dx download project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations.parquet -o {anno_dir}

anno_df = pl.scan_parquet(f'{anno_dir}/annotations.parquet')
anno_df.head().collect()

In [ ]:
# Download annotations fill_na file
!dx download project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fill_na.parquet -o {anno_dir}

anno_fillna_df = pl.scan_parquet(f'{anno_dir}/annotations_fill_na.parquet')
anno_fillna_df.head().collect()

## Drop abexp tissue columns

In [ ]:
ab_exp_cols = [c for c in anno_fillna_df.columns if c.startswith('abexp_')]

drop_abexp_cols = list(set(ab_exp_cols) - {'abexp_abs_max', 'abexp_abs_max_is_na'})
drop_abexp_cols += ['ac_ukb_eur', 'mac_ukb_eur', 'mac_ukb']

anno_fillna_df = anno_fillna_df.drop(drop_abexp_cols)
anno_fillna_df.columns

# Annotate UKB MAC and sample counts

In [ ]:
anno_fillna_df = (
    anno_fillna_df

    .join(
        varmeta_df.select(['id', 'sc_ukb', 'ac_ukb', 'mac_ukb']).lazy(), 
        on='id', 
        how='inner'
    )
)

# Annotate VEP coding regions

In [ ]:
vep_cds_relaxed = [
    # LoF variants
    'consequence_stop_gained',
    'consequence_frameshift_variant',

    # Missense and synonymous variants
    'consequence_synonymous_variant',
    'consequence_protein_altering_variant',
    'consequence_missense_variant',

    # Splicing variants
    'consequence_splice_region_variant',
    'consequence_splice_acceptor_variant',
    'consequence_splice_donor_variant',
    'consequence_splice_donor_region_variant',
    'consequence_splice_donor_5th_base_variant',
    'consequence_splice_polypyrimidine_tract_variant',

    # Other coding sequence variants
    'consequence_start_lost',
    'consequence_stop_lost',
    'consequence_start_retained_variant',
    'consequence_stop_retained_variant',
    'consequence_inframe_deletion',
    'consequence_inframe_insertion',
    'consequence_coding_sequence_variant',
]

anno_fillna_df = (
    anno_fillna_df
    .with_columns(
        vep_cds_relaxed = pl.sum_horizontal(vep_cds_relaxed) > 0,
    )
)

# Annotate protein domains

## Merge protein position, amino acid mutation

In [ ]:
anno_fillna_df = (
    anno_fillna_df
    
    .join(
        anno_df.select(['id', 'region', 'protein_position', 'amino_acids']).lazy(), 
        on='id', 
        how='inner'
    )
)

In [ ]:
# anno_fillna_df.sink_parquet(f'{anno_dir}/annotations_fillna_ukbgym_tmp.parquet', engine='streaming')

## Annotate protein domains

In [ ]:
anno_ukbgym_df = pl.scan_parquet(f'{anno_dir}/annotations_fillna_ukbgym_tmp.parquet')

anno_ukbgym_df.head().collect()

In [ ]:
anno_coding = (
    anno_ukbgym_df
    .select(['id', 'region', 'protein_position'])
    .drop_nulls()
    .with_columns(
        pos_raw = pl.col('protein_position').str.split('/').list.get(0)
    )
    .with_columns(
        split_struct = pl.col('pos_raw').str.split_exact('-', 1)
    )
    .with_columns(
        aa_start = pl.col('split_struct').struct.field('field_0').cast(pl.Int32, strict=False),
        aa_end = (
            pl.col('split_struct').struct.field('field_1')
            .fill_null(pl.col('split_struct').struct.field('field_0'))
            .cast(pl.Int32, strict=False)
        )
    )
    .drop(['pos_raw', 'split_struct'])
    .collect(engine='streaming')
)

anno_coding

In [ ]:
ginfo = (
    pl.read_parquet('PATH_TO_FILE')
    .filter(pl.col('ensembl_canonical') == True)
    .select(['gene_stable_id', 'gene_name', 'uniprotkb_gene_name_id', 'uniprotkb_gene_name_symbol', 'gene_type'])
    .rename({
        'gene_stable_id': 'region',
        'uniprotkb_gene_name_id': 'uniprot_id',
        'uniprotkb_gene_name_symbol': 'uniprot_name'
    })
    .filter(pl.col('gene_type') == 'protein_coding')
)
ginfo

## Disorder - MobiDB

In [ ]:
mobi_df_raw = pl.read_csv(
    "PATH_TO_FILE",
    separator="\t",
    has_header=False,
    new_columns=[
        "uniprot_id",
        "feature",
        "protein_regions",
        "disorder_content",
        "disorder_count",
        "length"
    ]
)

mobi_df_raw

In [ ]:
a = mobi_df_raw.filter(pl.col('uniprot_id')=='P04637').filter(pl.col('feature').str.contains('disorder'))
a

In [ ]:
mobi_df = (
    mobi_df_raw
    .join(ginfo.select(['uniprot_id', 'region']).unique(), on='uniprot_id', how='inner')  # Keep only proteins present in ginfo
    # .join(anno_coding.select(['region']).unique(), on='region', how='semi')  # Keep only proteins present in anno_coding

    .with_columns(pl.col("protein_regions").str.split(",")) # Split "1..2,71..75" into list
    .explode("protein_regions")                             # Create new row for each region
    .with_columns(
        # Split "1..2" into separate Start and End columns
        pl.col("protein_regions")
        .str.split_exact("..", 1)
        .struct.rename_fields(["domain_start", "domain_end"])
        .alias("protein_region_struct")
    )
    .unnest("protein_region_struct")
    .with_columns([
        pl.col("domain_start").cast(pl.Int64),
        pl.col("domain_end").cast(pl.Int64)
    ])

    .with_columns(
        mobi_feature_source = pl.col("feature").str.split("-").list.get(0),
        mobi_feature_type = pl.col("feature").str.split("-").list.get(1),
        mobi_feature_subtype = pl.col("feature").str.split("-").list.get(2)
    )

    .with_columns(
        mobi_curated_disorder_priority = (pl.col('mobi_feature_source')=='curated') & (pl.col('mobi_feature_type')=='disorder') & (pl.col('mobi_feature_subtype')=='priority'),
        mobi_lip_full = (pl.col('mobi_feature_type')=='lip') & (pl.col('mobi_feature_subtype')=='priority'),
    )

    .filter(
        (pl.col('mobi_curated_disorder_priority') == True) |
        (pl.col('mobi_lip_full') == True)
    )

    .select(['region', 'uniprot_id', 'domain_start', 'domain_end', 'mobi_curated_disorder_priority', 'mobi_lip_full'])
    .unique()
    .sort(['uniprot_id', 'domain_start'])
)

mobi_df

## Structured domains - TED

In [ ]:
ted_df_raw = pl.read_csv('PATH_TO_FILE', separator='\t', has_header=False)
ted_df_raw

In [ ]:
ted_df = (
    ted_df_raw
    .with_columns(
        uniprot_id = pl.col('column_1').str.split('-').list.get(1),
        ted_id = pl.col('column_1').str.slice(-5),
    )
    .with_columns(pl.col("column_4").str.split("_")) # Split "1-2_71-75" into list
    .explode("column_4")                             # Create new row for each region
    .with_columns(
        # Split "1-2" into separate Start and End columns
        pl.col("column_4")
        .str.split_exact("-", 1)
        .struct.rename_fields(["domain_start", "domain_end"])
        .alias("protein_region_struct")
    )
    .unnest("protein_region_struct")
    .with_columns(
        [
            pl.col("domain_start").cast(pl.Int64),
            pl.col("domain_end").cast(pl.Int64)
        ],
        ted_domain = pl.lit(True)
    )
    .select(['uniprot_id', 'domain_start', 'domain_end', 'ted_domain'])
    .unique()
    .join(ginfo.select(['uniprot_id', 'region']).unique(), on='uniprot_id', how='inner')  # Keep only proteins present in ginfo
)

ted_df

## Low complexity regions

In [ ]:
lc_df_raw = pl.read_csv('PATH_TO_FILE', separator='\t')
lc_df_raw

In [ ]:
lc_df = (
    lc_df_raw
    .filter(pl.col('Organism')=='H. sapiens')
    .rename({
        'Protein ID': 'uniprot_id',
    })
    .with_columns(
        # Split "1-2" into separate Start and End columns
        pl.col("Domain Boundaries")
        .str.strip_chars("()")
        .str.split_exact("-", 1)
        .struct.rename_fields(["domain_start", "domain_end"])
        .alias("protein_region_struct")
    )
    .unnest("protein_region_struct")
    .with_columns(
        [
            pl.col("domain_start").cast(pl.Int64),
            pl.col("domain_end").cast(pl.Int64)
        ],
        low_complexity_domain = pl.lit(True)
    )
    .select(['uniprot_id', 'domain_start', 'domain_end', 'low_complexity_domain'])
    .unique()
    .join(ginfo.select(['uniprot_id', 'region']).unique(), on='uniprot_id', how='inner')  # Keep only proteins present in ginfo
)

lc_df

## Merge with annotations

In [ ]:
all_df = pl.concat([mobi_df, ted_df, lc_df], how='diagonal').fill_null(False)
all_df

In [ ]:
overlap_df = (
    anno_coding.lazy() # Use lazy for better performance on large joins
    .join(
        all_df.lazy(), 
        on="region", 
        how="inner"
    )
    .filter(
        # (Variant Start <= Region End) AND (Variant Start >= Region Start)
        ((pl.col("aa_start") <= pl.col("domain_end")) & (pl.col("aa_start") >= pl.col("domain_start"))) |

        # (Variant End <= Region End) AND (Variant End >= Region Start)
        ((pl.col("aa_end") <= pl.col("domain_end")) & (pl.col("aa_end") >= pl.col("domain_start")))
    )
    .group_by(['id', 'region', 'aa_start', 'aa_end'])
    .agg([
        pl.col('mobi_curated_disorder_priority').max().alias('mobi_curated_disorder_priority'),
        pl.col('mobi_lip_full').max().alias('mobi_lip_full'),
        pl.col('ted_domain').max().alias('ted_domain'),
        pl.col('low_complexity_domain').max().alias('low_complexity_domain'),
    ])
    .collect()
)

overlap_df

In [ ]:
all_df = (
    anno_ukbgym_df.join(
        overlap_df.drop(['aa_start', 'aa_end']).lazy(),
        on=['id', 'region'],
        how='left'
    )
    .with_columns(
        pl.col('mobi_curated_disorder_priority').fill_null(False),
        pl.col('mobi_lip_full').fill_null(False),
        pl.col('ted_domain').fill_null(False),
        pl.col('low_complexity_domain').fill_null(False)
    )
    .unique()
    .collect(engine='streaming')
)

all_df.write_parquet(f'{anno_dir}/annotations_fillna_ukbgym.parquet')
all_df

In [ ]:
!dx upload {anno_dir}/annotations_fillna_ukbgym.parquet --path project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/

In [ ]:
anno_ukbgym_df.collect()